<a href="https://colab.research.google.com/github/p-tech/wbs-dm-2026/blob/main/web_scrape/website_crawl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin, urlparse

"""
WEB CRAWLING DEMONSTRATION: P-TECH.ORG.UK
-----------------------------------------
BENEFITS OF WEB CRAWLING:
1. Market Intelligence: Monitor competitor pricing and service updates.
2. Content Aggregation: Collect news or research into a single database.
3. Lead Generation: Identify potential partners or clients via profiles.
4. SEO Monitoring: Identify broken links and check search engine visibility.
5. Sentiment Analysis: Gather public reviews for brand perception.
"""

def is_internal(url, base_domain):
    """Checks if a URL belongs to the same domain."""
    return urlparse(url).netloc == base_domain

def crawl_site(start_url, max_pages=10):
    """
    Crawls multiple pages of a website starting from the start_url.
    Follows internal links until max_pages is reached.
    """
    base_domain = urlparse(start_url).netloc
    to_visit = [start_url]
    visited = set()
    all_data = []

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    print(f"--- Starting Multi-Page Crawl (Limit: {max_pages} pages) ---")

    while to_visit and len(visited) < max_pages:
        current_url = to_visit.pop(0)

        if current_url in visited:
            continue

        print(f"\n[Crawling {len(visited) + 1}/{max_pages}]: {current_url}")

        try:
            response = requests.get(current_url, headers=headers, timeout=10)
            visited.add(current_url)

            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')

                # Extract Page Info
                title = soup.title.string.strip() if soup.title else "No Title"

                # Find all links
                links_found = []
                for a in soup.find_all('a', href=True):
                    # Convert relative links to absolute
                    absolute_link = urljoin(current_url, a['href']).split('#')[0].rstrip('/')

                    # If it's internal and not visited, add to queue
                    if is_internal(absolute_link, base_domain) and absolute_link not in visited:
                        if absolute_link not in to_visit:
                            to_visit.append(absolute_link)

                    links_found.append(absolute_link)

                all_data.append({
                    'URL': current_url,
                    'Title': title,
                    'Links Found': len(links_found)
                })

                # Best Practice: Be polite, wait between requests
                time.sleep(1)
            else:
                print(f"[-] Failed to retrieve: {response.status_code}")

        except Exception as e:
            print(f"[-] Error on {current_url}: {e}")

    return all_data

def main():
    target_url = "https://www.p-tech.org.uk"
    # We set a small limit of 5 pages for this demonstration
    crawled_results = crawl_site(target_url, max_pages=5)

    if crawled_results:
        print("\n--- Business Analysis Summary ---")
        df = pd.DataFrame(crawled_results)
        print(df.to_string(index=False))

        print("\nBEST PRACTICES & ETHICS:")
        print("- Always check the site's /robots.txt file.")
        print("- Use time.sleep() (Politeness Delay) to avoid server strain.")
        print("- Limit depth/page count to avoid 'Spider Traps' or infinite loops.")
    else:
        print("\nNo data was collected.")

if __name__ == "__main__":
    main()

--- Starting Multi-Page Crawl (Limit: 5 pages) ---

[Crawling 1/5]: https://www.p-tech.org.uk

[Crawling 2/5]: https://www.p-tech.org.uk/about-us/p-tech
[-] Error on https://www.p-tech.org.uk/about-us/p-tech: HTTPSConnectionPool(host='www.p-tech.org.uk', port=443): Read timed out. (read timeout=10)

[Crawling 2/5]: https://www.p-tech.org.uk/about-us/clients

[Crawling 3/5]: https://www.p-tech.org.uk/about-us/staff/james-pennington

[Crawling 4/5]: https://www.p-tech.org.uk/consultancy-2

[Crawling 5/5]: https://www.p-tech.org.uk/consultancy/digital-marketing

--- Business Analysis Summary ---
                                                      URL                                                                                                                             Title  Links Found
                                https://www.p-tech.org.uk                         Digital Marketing and IT Consultancy, Digital Marketing Training, Coventry - Warwickshire - West Midlands          11

# Page Analysis

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from collections import Counter
from urllib.parse import urljoin, urlparse

"""
WEB CRAWLING DEMONSTRATION: P-TECH.ORG.UK
-----------------------------------------
BENEFITS OF WEB CRAWLING:
1. Market Intelligence: Monitor competitor pricing and service updates.
2. Content Aggregation: Collect news or research into a single database.
3. Lead Generation: Identify potential partners or clients via profiles.
4. SEO Monitoring: Identify broken links and check search engine visibility.
5. Sentiment Analysis: Gather public reviews for brand perception.
6. Keyword Analysis (N-grams): Identify core themes and SEO focus.
"""

def get_ngrams(text, n=1):
    """
    Simple function to generate n-grams from text.
    Cleans text, removes short words/common stopword-like terms, and counts frequencies.
    """
    # Clean text: remove non-alphanumeric, lowercase, and split
    words = re.findall(r'\b\w\w+\b', text.lower())

    # Optional: Basic filtering of very common web words
    stop_words = {'the', 'and', 'for', 'with', 'this', 'that', 'from', 'are', 'was', 'were'}
    words = [w for w in words if w not in stop_words]

    if n == 1:
        return words

    # Create bigrams/trigrams by zipping the word list with its own offset
    return [" ".join(gram) for gram in zip(*[words[i:] for i in range(n)])]

def extract_keywords(soup, top_n=3):
    """Extracts top unigrams and bigrams to find the page focus."""
    # Get all visible text from the body
    page_text = soup.body.get_text(separator=' ') if soup.body else ""

    unigrams = get_ngrams(page_text, n=1)
    bigrams = get_ngrams(page_text, n=2)

    top_unigrams = [item[0] for item in Counter(unigrams).most_common(top_n)]
    top_bigrams = [item[0] for item in Counter(bigrams).most_common(top_n)]

    return {
        'Keywords': ", ".join(top_unigrams),
        'Key Phrases': ", ".join(top_bigrams)
    }

def is_internal(url, base_domain):
    """Checks if a URL belongs to the same domain."""
    return urlparse(url).netloc == base_domain

def crawl_site(start_url, max_pages=5):
    """
    Crawls multiple pages and performs keyword focus analysis using N-grams.
    """
    base_domain = urlparse(start_url).netloc
    to_visit = [start_url]
    visited = set()
    all_data = []

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    print(f"--- Starting Multi-Page Crawl with Keyword Analysis (Limit: {max_pages} pages) ---")

    while to_visit and len(visited) < max_pages:
        current_url = to_visit.pop(0)

        if current_url in visited:
            continue

        print(f"\n[Crawling {len(visited) + 1}/{max_pages}]: {current_url}")

        try:
            response = requests.get(current_url, headers=headers, timeout=10)
            visited.add(current_url)

            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')

                # Extract Page Title
                title = soup.title.string.strip() if soup.title else "No Title"

                # Extract N-gram Focus
                focus_data = extract_keywords(soup)

                # Find all internal links for further crawling
                links_found = 0
                for a in soup.find_all('a', href=True):
                    absolute_link = urljoin(current_url, a['href']).split('#')[0].rstrip('/')
                    links_found += 1

                    if is_internal(absolute_link, base_domain) and absolute_link not in visited:
                        if absolute_link not in to_visit:
                            to_visit.append(absolute_link)

                # Combine crawl data with keyword focus
                page_info = {
                    'URL': current_url,
                    'Title': title[:40] + "...",
                    'Top Keywords': focus_data['Keywords'],
                    'Top Phrases': focus_data['Key Phrases']
                }
                all_data.append(page_info)

                time.sleep(1)
            else:
                print(f"[-] Failed to retrieve: {response.status_code}")

        except Exception as e:
            print(f"[-] Error on {current_url}: {e}")

    return all_data

def main():
    target_url = "https://www.p-tech.org.uk"
    crawled_results = crawl_site(target_url, max_pages=5)

    if crawled_results:
        print("\n--- Content Focus & Keyword Analysis ---")
        df = pd.DataFrame(crawled_results)
        # Displaying with adjusted width for readability
        pd.set_option('display.max_colwidth', 50)
        print(df.to_string(index=False))

        print("\nBENEFITS OF THIS ANALYSIS:")
        print("- Automated SEO Audits: Quickly identify if pages are optimized for the right terms.")
        print("- Content Categorization: Automatically tag pages based on their primary topics.")
        print("- Trend Detection: Spot recurring themes across an entire corporate domain.")
    else:
        print("\nNo data was collected.")

if __name__ == "__main__":
    main()

--- Starting Multi-Page Crawl with Keyword Analysis (Limit: 5 pages) ---

[Crawling 1/5]: https://www.p-tech.org.uk

[Crawling 2/5]: https://www.p-tech.org.uk/about-us/p-tech
[-] Error on https://www.p-tech.org.uk/about-us/p-tech: HTTPSConnectionPool(host='www.p-tech.org.uk', port=443): Read timed out. (read timeout=10)

[Crawling 2/5]: https://www.p-tech.org.uk/about-us/clients

[Crawling 3/5]: https://www.p-tech.org.uk/about-us/staff/james-pennington

[Crawling 4/5]: https://www.p-tech.org.uk/consultancy-2

[Crawling 5/5]: https://www.p-tech.org.uk/consultancy/digital-marketing

--- Content Focus & Keyword Analysis ---
                                                      URL                                       Title                 Top Keywords                                    Top Phrases
                                https://www.p-tech.org.uk Digital Marketing and IT Consultancy, Di...  digital, marketing, website digital marketing, social media, search engine
               